In [1]:
import numpy as np
import pandas as pd


In [2]:
online = pd.read_csv("thetaPSD_online_2back_-1580742246_20260603_113547.csv")
online

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,Ch06
0,0.000,-23045.730469,-0.037059,0.001373,0.000027,-0.555017,143.123611
1,0.004,-22165.824219,-0.321645,0.103456,0.002097,-0.554522,8185.798340
2,0.008,-21761.943359,-1.411098,1.991199,0.041921,-0.544995,17259.072266
3,0.012,-21871.234375,-4.269454,18.228237,0.406485,-0.457779,165271.500000
4,0.016,-22129.322266,-10.180685,103.646347,2.479412,0.038137,228450.125000
...,...,...,...,...,...,...,...
8700,34.800,-22648.527344,-0.123552,0.015265,15.943089,3.259113,3.318092
8701,34.804,-22711.730469,-0.556624,0.309831,15.635612,3.185553,3.318092
8702,34.808,-23103.214844,-0.971354,0.943528,15.272611,3.098711,3.318092
8703,34.812,-23091.365234,-1.364592,1.862112,14.861465,3.000350,3.318092


In [3]:
online = online.rename(columns={"Time": "Time", " Ch01": "Raw", " Ch02": "Theta Bandpass", " Ch03": "Power", " Ch04": "Moving Average", " Ch05": "Z-score", " Ch06": "Hold"})
online

,Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold
0,0.000,-23045.730469,-0.037059,0.001373,0.000027,-0.555017,143.123611
1,0.004,-22165.824219,-0.321645,0.103456,0.002097,-0.554522,8185.798340
2,0.008,-21761.943359,-1.411098,1.991199,0.041921,-0.544995,17259.072266
3,0.012,-21871.234375,-4.269454,18.228237,0.406485,-0.457779,165271.500000
4,0.016,-22129.322266,-10.180685,103.646347,2.479412,0.038137,228450.125000
...,...,...,...,...,...,...,...
8700,34.800,-22648.527344,-0.123552,0.015265,15.943089,3.259113,3.318092
8701,34.804,-22711.730469,-0.556624,0.309831,15.635612,3.185553,3.318092
8702,34.808,-23103.214844,-0.971354,0.943528,15.272611,3.098711,3.318092
8703,34.812,-23091.365234,-1.364592,1.862112,14.861465,3.000350,3.318092


In [4]:
threshold_mask= online["Hold"] > 1.5
threshold_crossed = online[threshold_mask]
threshold_crossed[720:]

,Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold
720,2.880,-21722.509766,-6.363634,40.495838,16.401749,3.368839,3.278100
721,2.884,-21975.162109,-6.670848,44.500206,16.913946,3.491374,3.278100
722,2.888,-22214.605469,-6.868125,47.171135,17.533382,3.639565,3.278100
723,2.892,-21840.884766,-6.952446,48.336506,18.229477,3.806095,3.278100
724,2.896,-21702.718750,-6.922988,47.927757,18.968912,3.982993,3.278100
...,...,...,...,...,...,...,...
8700,34.800,-22648.527344,-0.123552,0.015265,15.943089,3.259113,3.318092
8701,34.804,-22711.730469,-0.556624,0.309831,15.635612,3.185553,3.318092
8702,34.808,-23103.214844,-0.971354,0.943528,15.272611,3.098711,3.318092
8703,34.812,-23091.365234,-1.364592,1.862112,14.861465,3.000350,3.318092


In [24]:
merged_df = online.loc[online["Time"] > 10.00].copy()
merged_df

,Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold
2501,10.004,-22733.546875,0.665044,0.442283,2.958623,0.152781,0.150709
2502,10.008,-22419.121094,1.319772,1.741797,2.993124,0.161034,0.150709
2503,10.012,-22187.880859,1.966396,3.866713,3.069606,0.179331,0.150709
2504,10.016,-22333.697266,2.589395,6.704967,3.202396,0.211099,0.150709
2505,10.020,-22723.843750,3.173574,10.071569,3.402312,0.258926,0.150709
...,...,...,...,...,...,...,...
8700,34.800,-22648.527344,-0.123552,0.015265,15.943089,3.259113,3.318092
8701,34.804,-22711.730469,-0.556624,0.309831,15.635612,3.185553,3.318092
8702,34.808,-23103.214844,-0.971354,0.943528,15.272611,3.098711,3.318092
8703,34.812,-23091.365234,-1.364592,1.862112,14.861465,3.000350,3.318092


In [25]:
buffer = []
offline_z = []
last_theta_val = None

MU = 2.32
SIGMA = 4.18
MAD_THRESHOLD = 6
THETA_THRESHOLD_Z = 1.5
INITIAL_CUTOFF = 100.0
BUFFER_SIZE = 500
COOLDOWN_TIME = 5

# Ch05 = index 4
channel = "Hold"

for i in range(len(merged_df)):

    theta_val = merged_df[channel].iloc[i]   # MUST match online
    if last_theta_val is not None and theta_val == last_theta_val:
        offline_z.append(np.nan) 
        continue
    last_theta_val = theta_val
    # Warm-up
    if len(buffer) <= 450:
        if theta_val < INITIAL_CUTOFF:
            buffer.append(theta_val)
        offline_z.append(np.nan)
        continue

    # Rolling buffer
    if len(buffer) > BUFFER_SIZE:
        buffer.pop(0)

    arr = np.array(buffer)
    median = np.median(arr)
    mad = np.median(np.abs(arr - median)) + 1e-6

    # Artifact rejection
    z_art = abs(theta_val - median) / mad
    if z_art > MAD_THRESHOLD:
        offline_z.append(np.nan)
        continue

    # Clean sample
    buffer.append(theta_val)

    # Z-score
    theta_z = abs(theta_val - MU) / SIGMA
    offline_z.append(theta_z)

merged_df["offline z-score"] = offline_z


In [26]:
LIFU = np.zeros(len(merged_df))
last_trigger_time = -999

for i in range(len(merged_df)):
    theta_z = merged_df["Hold"].iloc[i]
    now = merged_df["Time"].iloc[i]

    if theta_z > THETA_THRESHOLD_Z and theta_z < MAD_THRESHOLD and (now - last_trigger_time > COOLDOWN_TIME):
        LIFU[i] = 1.0
        last_trigger_time = now


In [27]:
merged_df["LIFU"] = LIFU
merged_df

,Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,offline z-score,LIFU
2501,10.004,-22733.546875,0.665044,0.442283,2.958623,0.152781,0.150709,NaN,0.0
2502,10.008,-22419.121094,1.319772,1.741797,2.993124,0.161034,0.150709,NaN,0.0
2503,10.012,-22187.880859,1.966396,3.866713,3.069606,0.179331,0.150709,NaN,0.0
2504,10.016,-22333.697266,2.589395,6.704967,3.202396,0.211099,0.150709,NaN,0.0
2505,10.020,-22723.843750,3.173574,10.071569,3.402312,0.258926,0.150709,NaN,0.0
...,...,...,...,...,...,...,...,...,...
8700,34.800,-22648.527344,-0.123552,0.015265,15.943089,3.259113,3.318092,NaN,0.0
8701,34.804,-22711.730469,-0.556624,0.309831,15.635612,3.185553,3.318092,NaN,0.0
8702,34.808,-23103.214844,-0.971354,0.943528,15.272611,3.098711,3.318092,NaN,0.0
8703,34.812,-23091.365234,-1.364592,1.862112,14.861465,3.000350,3.318092,NaN,0.0


In [31]:
mask_lifu = merged_df["LIFU"] == 1.0
lifu_on = merged_df[mask_lifu]
lifu_on["Time"] = lifu_on["Time"] + 0.5
lifu_on

C:\Users\jshin\AppData\Local\Temp\ipykernel_24464\3937477043.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lifu_on["Time"] = lifu_on["Time"] + 0.5


,Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,offline z-score,LIFU
2529,10.616,-22344.853516,-3.951867,15.617251,9.437488,1.702748,1.702748,NaN,1.0
4529,18.616,-22551.037109,-6.931189,48.041382,10.539940,1.966493,1.966493,NaN,1.0
6139,25.056,-23039.271484,4.978577,24.786224,11.810321,2.270412,2.270412,NaN,1.0
7419,30.176,-23008.494141,-3.454917,11.936450,8.849416,1.562061,1.562061,0.181325,1.0
8669,35.176,-23236.349609,-1.776081,3.154463,17.674408,3.673303,3.673303,NaN,1.0
